# Neighborhood-level approach

## Library imports

Load modules

In [ ]:
# % load nb_black

Load libraries

In [ ]:
import warnings
import random
import sys
import scanpy as sc
import numpy as np
import pandas as pd
import os
import time
import multiprocessing

from functools import partial

# Set dir paths
label_indpt_dir = os.path.abspath(os.getcwd()) # mapqc_l_indpt
notebooks_dir = os.path.abspath(os.pardir) # notebooks
mapqc_dir = os.path.abspath(os.path.join(notebooks_dir, os.pardir)) # MapQC

# Update paths
sys.path.append(notebooks_dir)
sys.path.append(label_indpt_dir)
sys.path.append(mapqc_dir)

from pvalues_label_indpt import get_pvalues

## General settings

Set working directories

In [ ]:
# Set global seed
np.random.seed(42)

# Ignore warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

# Set directories (change as you like)
DATA_DIR = os.path.abspath(os.path.join(mapqc_dir, "data", "HLCA_full_tranf.h5ad"))

os.makedirs(os.path.join(label_indpt_dir, "Results"), exist_ok=True)
RESULTS_DIR = os.path.abspath(os.path.join(label_indpt_dir, "Results"))

os.makedirs(os.path.join(RESULTS_DIR, "Neighborhood_based_HLCA"), exist_ok=True)
HCLA_RES_DIR = os.path.abspath(os.path.join(RESULTS_DIR, "Neighborhood_based_HLCA"))

### Hyperparameter settings

In [ ]:
# Change as you like
K_RANGE = [
    500, 2000, 5000, 8000, 
    10000, 30000, 50000, 75000, 100000
]
PROP_CELLS = [50, 100, 500, 1000]

## Functions definition

In [ ]:
def save_pvalues(args):
    query_id, query_path, k, sample_freq, run_id, condition, filter_samples, filter_nhoods = args
    
    output_file = f"{HCLA_RES_DIR}/{query_path}/pvalues_{k}_{sample_freq}_{run_id + 1}.csv"
    
    # Check if output file already exists
    if os.path.exists(output_file):
        print(f"Skipping {output_file} as it already exists.")
        return
    
    try:
        res, neighborhood_stats = get_pvalues(
            adata_subset[query_id], 
            k, 
            sample_freq, 
            condition, 
            filter_samples=filter_samples, 
            filter_nhoods=filter_nhoods
        )
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        neighborhood_stats.to_csv(output_file)
    except ValueError as e:
        print(e)

## Data Loading

In [ ]:
adata_ext_full = sc.read_h5ad(DATA_DIR)

## Differential mixing analysis

In [ ]:
adata_ext_full.obs.dataset.cat.categories

In [ ]:
# Change as you like
query_datasets = [
    "Meyer_2021_3prime", 
    "Meyer_2021_5prime", 
    "Tata_unpubl", 
    "Duong_lungMAP_unpubl"
]

Select lung conditions on query

In [ ]:
for ds in query_datasets:
    print(
        f"Dataset: {ds} \n {adata_ext_full[adata_ext_full.obs.dataset == ds].obs.lung_condition.cat.categories}"
    )
    

Set paths to HLCA query datasets

In [ ]:
path_queries = {}
for ds in query_datasets:
    path_queries[f"{ds}_unfiltered"] = (f"{ds}/unfiltered/", "Healthy")
    path_queries[f"{ds}_filtered"] = (f"{ds}/filtered/", "Healthy")
path_queries

Create directories

In [ ]:
for path, _ in path_queries.values():
    full_path = os.path.join(HCLA_RES_DIR, path)
    os.makedirs(full_path, exist_ok=True)

Data subsetting

In [ ]:
adata_subset = {
    f"{ds}": adata_ext_full[
        (adata_ext_full.obs.core_or_extension == "core")
        |(adata_ext_full.obs.dataset == ds)
    ].copy()
    for ds in query_datasets
}
adata_subset

Get pvalues

In [ ]:
query_params = [
    (path.split('/')[0], path, condition, "unfiltered" not in path, "unfiltered" not in path)
    for path, condition in path_queries.values()
]
query_params

In [ ]:
# Pool worker processeses
num_cores = multiprocessing.cpu_count() #56
pool = multiprocessing.Pool(processes=num_cores)

# List arguments for parallel processing
args_list = [(query_id, query_path, k, sample_freq, run_id, condition, filter_samples, filter_nhoods)
             for k in K_RANGE
             for sample_freq in PROP_CELLS
             for run_id in range(10)
             for query_id, query_path, condition, filter_samples, filter_nhoods in query_params]

# Parallelize
pool.map(save_pvalues, args_list)
pool.close()
pool.join()